# Phase 3 — Feature Analysis

## Objective

Transform validated churn patterns from Phase 2 into customer-level analytical
features that support customer segmentation, retention prioritisation, and
revenue exposure analysis.

## Business Context

Phase 2 identified meaningful differences in observed churn across tenure,
contract type, internet service, payment method, and monthly charges.

Phase 3 converts selected patterns into reproducible analytical features.

## Analytical Grain

One row = one customer.

## Input Dataset

`data/processed/telco_cleaned.csv`

## Output Dataset

`data/processed/telco_features.csv`

## Features Created

1. `tenure_band`
2. `spend_tier`
3. `is_at_risk`
4. `revenue_at_risk`

## Supporting Analysis

`historical_risk_segment` is used to validate the selected risk criteria
against historical churn. It is not treated as a final feature.

## Important Limitation

These features are analytical segmentation variables. They do not establish
causality and `is_at_risk` is not a predictive churn model.

## 1. Imports

In [1]:
import pandas as pd
import numpy as np

## 2. Load & Validate Data

In [2]:
df = pd.read_csv("../data/processed/telco_cleaned.csv")

print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
# Dataset integrity checks
assert df.shape == (7043, 21), f"Unexpected shape: {df.shape}"
assert df["customerID"].is_unique, "customerID is not unique"
assert df["customerID"].notna().all(), "customerID contains nulls"

print("Dataset integrity checks passed.")

Dataset integrity checks passed.


In [4]:
# Grain validation: one row per customer
assert df["customerID"].nunique() == len(df), "Grain violation: duplicate customers detected"

print("Grain validation passed: one row per customer.")

Grain validation passed: one row per customer.


In [5]:
# Baseline: overall churn rate (used for risk-lift calculation later)
overall_churn_rate = df["Churn"].eq("Yes").mean() * 100

churn_count = df["Churn"].value_counts()
print(f"Churned customers:  {churn_count['Yes']:,}")
print(f"Retained customers: {churn_count['No']:,}")
print(f"Overall churn rate: {overall_churn_rate:.2f}%")

Churned customers:  1,869
Retained customers: 5,174
Overall churn rate: 26.54%


## 3. Phase 2 → Phase 3 Analytical Bridge

## Why These Features?

Phase 2 identified several customer characteristics associated with different
observed churn rates.

The following characteristics were selected for deeper feature analysis:

- Customer tenure
- Monthly charges
- Contract type
- Internet service
- Payment method

These were selected because they showed meaningful differences in observed
churn and can be translated into interpretable customer segments.

The features created in this phase are therefore derived from the analytical
findings of Phase 2 rather than selected arbitrarily.

## 4. Reusable Helpers

In [6]:
def validate_feature(series, feature_name):
    """Print a consistent validation summary for a feature column."""
    print(f"--- {feature_name} Validation ---")
    print(f"  Nulls:         {series.isna().sum()}")
    print(f"  Unique values: {series.nunique()}")
    print(f"  Rows:          {len(series)}")
    print()


def churn_summary(data, group_col):
    """Return customer count and observed churn rate by group."""
    return (
        data.groupby(group_col, observed=True)
            .agg(
                customer_count=("customerID", "count"),
                churn_rate=("Churn", lambda x: (x == "Yes").mean() * 100)
            )
            .round(2)
            .reset_index()
    )

## 5. Feature 1 — `tenure_band`

### Definition

`tenure_band` converts the continuous `tenure` variable into six discrete
customer lifecycle segments using fixed 10-month intervals.

### Why this feature?

The churn rate shows a clear decreasing pattern as customer tenure increases.
Converting tenure into interpretable bands makes it easier to communicate
lifecycle-stage retention patterns to business stakeholders and supports
categorical downstream analysis.

### Implementation

In [7]:
tenure_bins = [0, 10, 20, 30, 40, 50, np.inf]
tenure_labels = ["0-10", "10-20", "20-30", "30-40", "40-50", "50+"]

df["tenure_band"] = pd.cut(
    df["tenure"],
    bins=tenure_bins,
    labels=tenure_labels,
    right=False
)

### Validation

In [8]:
assert df["tenure_band"].notna().all(), "tenure_band contains nulls"

validate_feature(df["tenure_band"], "tenure_band")
df["tenure_band"].value_counts().sort_index()

--- tenure_band Validation ---
  Nulls:         0
  Unique values: 6
  Rows:          7043



tenure_band
0-10     1854
10-20     953
20-30     762
30-40     653
40-50     648
50+      2173
Name: count, dtype: int64

### Churn Analysis by Tenure Band

In [9]:
tenure_summary = churn_summary(df, "tenure_band")
tenure_summary

,tenure_band,customer_count,churn_rate
0,0-10,1854,49.78
1,10-20,953,32.53
2,20-30,762,23.10
3,30-40,653,22.05
4,40-50,648,18.21
5,50+,2173,9.11


### Interpretation

The churn rate shows a clear decreasing pattern as customer tenure increases.

- **0–10 months:** 49.78% churn (1,854 customers) — highest-risk cohort.
- **10–20 months:** 32.53% churn (953 customers).
- **20–30 months:** 23.10% churn (762 customers).
- **30–40 months:** 22.05% churn (653 customers).
- **40–50 months:** 18.21% churn (648 customers).
- **50+ months:** 9.11% churn (2,173 customers) — most stable cohort.

The difference between the lowest- and highest-tenure segments is **40.67 percentage points**.

### Limitation

The boundaries are analytical groupings, not officially defined business lifecycle stages.
Customers close to a boundary may fall into different bands despite similar tenure.
`tenure_band` should be treated as an **analytical segmentation**, not as evidence
that a specific tenure threshold causes churn.

## 6. Feature 2 — `spend_tier`

### Definition

`spend_tier` converts `MonthlyCharges` into five discrete spending segments
using 20-unit intervals, with an open-ended `90+` category.

### Why this feature?

Monthly charges vary widely across customers. Converting charges to tiers
makes it easier to compare churn across spending ranges and identify
high-value segments with elevated churn exposure.

### Implementation

In [10]:
spend_bins = [10, 30, 50, 70, 90, np.inf]
spend_labels = ["10-30", "30-50", "50-70", "70-90", "90+"]

df["spend_tier"] = pd.cut(
    df["MonthlyCharges"],
    bins=spend_bins,
    labels=spend_labels,
    right=False
)

# Note: observed minimum MonthlyCharges is 18.25, so the 10-30 bin captures
# all low-spend customers correctly.

### Validation

In [11]:
assert df["spend_tier"].notna().all(), "spend_tier contains nulls"

validate_feature(df["spend_tier"], "spend_tier")
df["spend_tier"].value_counts().sort_index()

--- spend_tier Validation ---
  Nulls:         0
  Unique values: 5
  Rows:          7043



spend_tier
10-30    1653
30-50     641
50-70    1158
70-90    1847
90+      1744
Name: count, dtype: int64

### Churn Analysis by Spend Tier

In [12]:
spend_summary = churn_summary(df, "spend_tier")
spend_summary

,spend_tier,customer_count,churn_rate
0,10-30,1653,9.80
1,30-50,641,31.05
2,50-70,1158,20.21
3,70-90,1847,37.95
4,90+,1744,32.86


### Interpretation

Churn varies considerably across spending tiers but is **not monotonically increasing**:

| Spend Tier | Customer Count | Churn Rate |
|------------|---------------|------------|
| 10–30      | 1,653         | 9.80%      |
| 30–50      | 641           | 31.05%     |
| 50–70      | 1,158         | 20.21%     |
| 70–90      | 1,847         | 37.95%     |
| 90+        | 1,744         | 32.86%     |

The 70–90 tier has the highest observed churn at 37.95%; the 10–30 tier has the lowest at 9.80%.

### Limitation

Spending thresholds are analytical groupings, not official pricing segments.
The non-monotonic pattern suggests that monthly charges alone do not explain churn —
other factors such as contract type and internet service likely interact with spending.
`spend_tier` should not be interpreted as evidence that higher charges directly cause churn.

## 7. Historical Risk Validation

### Two Analytical Concepts — Kept Separate

#### Historical Risk Validation

`historical_risk_segment`

**Purpose:** Evaluate whether the selected characteristics were historically
associated with elevated churn. Includes both churned and retained customers
who matched the criteria.

#### Current At-Risk Population

`is_at_risk`

**Purpose:** Identify currently **active** customers (Churn == "No") who match
those characteristics and represent future retention opportunities.

This distinction is essential: already-churned customers are excluded from
the current revenue exposure calculation.

### Historical Risk Segment — Definition & Construction

In [13]:
# Criteria selected from Phase 2 patterns:
# Month-to-month contract + short tenure + Fiber optic + Electronic check + mid-high charges
historical_risk_mask = (
    (df["Contract"] == "Month-to-month") &
    (df["tenure"] <= 12) &
    (df["InternetService"] == "Fiber optic") &
    (df["PaymentMethod"] == "Electronic check") &
    (df["MonthlyCharges"].between(50, 100, inclusive="both"))
)

historical_risk_segment = df[historical_risk_mask]

print(f"Historical risk segment size: {len(historical_risk_segment):,}")

Historical risk segment size: 596


### Validation — Observed Churn Rate

In [14]:
historical_churn_rate = (historical_risk_segment["Churn"] == "Yes").mean() * 100

historical_risk_comparison = pd.DataFrame({
    "group": ["Overall Population", "Remaining Customers", "Historical Risk Segment"],
    "customer_count": [
        len(df),
        len(df) - len(historical_risk_segment),
        len(historical_risk_segment)
    ],
    "churn_rate_pct": [
        round(overall_churn_rate, 2),
        round((df.loc[~historical_risk_mask, "Churn"] == "Yes").mean() * 100, 2),
        round(historical_churn_rate, 2)
    ]
})

historical_risk_comparison

,group,customer_count,churn_rate_pct
0,Overall Population,7043,26.54
1,Remaining Customers,6447,22.44
2,Historical Risk Segment,596,70.81


### Risk Lift

In [15]:
risk_lift = historical_churn_rate / overall_churn_rate
risk_difference_vs_overall = historical_churn_rate - overall_churn_rate

remaining_churn_rate = (df.loc[~historical_risk_mask, "Churn"] == "Yes").mean() * 100
risk_difference_vs_remaining = historical_churn_rate - remaining_churn_rate

print(f"Overall churn rate:                  {overall_churn_rate:.2f}%")
print(f"Historical risk segment churn rate:  {historical_churn_rate:.2f}%")
print(f"Risk lift (vs overall):              {risk_lift:.2f}x")
print(f"Risk difference vs overall:          +{risk_difference_vs_overall:.2f} pp")
print(f"Risk difference vs remaining:        +{risk_difference_vs_remaining:.2f} pp")

Overall churn rate:                  26.54%
Historical risk segment churn rate:  70.81%
Risk lift (vs overall):              2.67x
Risk difference vs overall:          +44.27 pp
Risk difference vs remaining:        +48.36 pp


### Interpretation

The historical risk segment has a churn rate approximately **2.67× higher** than the
overall population, and **~48 percentage points higher** than the remaining customer base.

This validates that the selected combination of criteria (Month-to-month contract,
short tenure, Fiber optic service, Electronic check, mid-high charges) was historically
associated with materially elevated churn.

### Limitation

This is a retrospective validation, not a causal model. The segment criteria were
identified through Phase 2 EDA and validated here against historical outcomes.
The same criteria are used below to identify **current** at-risk customers — those
who have not yet churned and may represent retention opportunities.

## 8. Feature 3 — `is_at_risk`

### Definition

`is_at_risk` is a binary indicator (0/1) identifying **currently active customers**
who match all five risk criteria from the historical validation above.

Customers who have already churned are explicitly excluded (Churn == "No" filter).

### Implementation

In [16]:
at_risk_mask = (
    (df["Contract"] == "Month-to-month") &
    (df["tenure"] <= 12) &
    (df["InternetService"] == "Fiber optic") &
    (df["PaymentMethod"] == "Electronic check") &
    (df["MonthlyCharges"].between(50, 100, inclusive="both")) &
    (df["Churn"] == "No")   # active customers only
)

df["is_at_risk"] = at_risk_mask.astype(int)

### Validation

In [17]:
# All is_at_risk == 1 customers must be active (Churn == "No")
assert df.loc[df["is_at_risk"] == 1, "Churn"].eq("No").all(),     "is_at_risk contains churned customers"

# Feature must be binary
assert df["is_at_risk"].isin([0, 1]).all(), "is_at_risk contains non-binary values"

# Dataset-specific checkpoint (fixed dataset)
assert df["is_at_risk"].sum() == 174,     f"Unexpected at-risk count: {df['is_at_risk'].sum()}"

validate_feature(df["is_at_risk"], "is_at_risk")

active_risk_summary = df.groupby("is_at_risk").agg(
    customer_count=("customerID", "count"),
    active_customers=("Churn", lambda x: (x == "No").sum())
)
active_risk_summary

--- is_at_risk Validation ---
  Nulls:         0
  Unique values: 2
  Rows:          7043



,customer_count,active_customers
is_at_risk,,
0,6869,5000
1,174,174


### Interpretation

174 currently active customers (2.47% of the base) match all five risk criteria.
These customers represent the highest-priority retention target identified by this analysis.

### Limitation

`is_at_risk` is a rule-based segment, not a predictive model. It identifies
customers who historically resembled a high-churn group; it does not predict
the probability that any individual customer will churn.

## 9. Feature 4 — `revenue_at_risk`

### Definition

`revenue_at_risk` is the monthly recurring charge (`MonthlyCharges`) for each
at-risk customer, and zero for all other customers.

This creates a **customer-level feature** that can be aggregated to produce the
**business-level KPI**: total current monthly recurring revenue exposure.

### Implementation

In [18]:
df["revenue_at_risk"] = np.where(
    df["is_at_risk"] == 1,
    df["MonthlyCharges"],
    0
)

### Validation

In [19]:
assert df["revenue_at_risk"].isna().sum() == 0, "revenue_at_risk contains nulls"
assert (df.loc[df["is_at_risk"] == 0, "revenue_at_risk"] == 0).all(),     "Non-at-risk customers have non-zero revenue_at_risk"

validate_feature(df["revenue_at_risk"], "revenue_at_risk")

total_revenue_at_risk = df["revenue_at_risk"].sum()
print(f"Monthly revenue at risk: {total_revenue_at_risk:,.2f}")

--- revenue_at_risk Validation ---
  Nulls:         0
  Unique values: 138
  Rows:          7043

Monthly revenue at risk: 13,933.80


### Business KPI Summary

In [20]:
revenue_risk_summary = pd.DataFrame({
    "metric": [
        "At-risk customers",
        "At-risk customer share (%)",
        "Monthly revenue at risk",
        "Average monthly charge per at-risk customer"
    ],
    "value": [
        int(df["is_at_risk"].sum()),
        round(df["is_at_risk"].mean() * 100, 2),
        round(df["revenue_at_risk"].sum(), 2),
        round(df.loc[df["is_at_risk"] == 1, "MonthlyCharges"].mean(), 2)
    ]
})

revenue_risk_summary

,metric,value
0,At-risk customers,174.00
1,At-risk customer share (%),2.47
2,Monthly revenue at risk,13933.80
3,Average monthly charge per at-risk customer,80.08


### Interpretation

The 174 at-risk customers represent **13,933.80 in current monthly recurring
revenue exposure** — revenue that may be lost if these customers churn.

This figure represents **potential monthly revenue exposure**, not lost revenue.
These customers have not churned yet and remain active.

### Limitation

Revenue exposure is calculated from `MonthlyCharges` only and does not include
additional revenue sources such as phone services, add-ons, or annual contract value.
The figure should be treated as a lower-bound estimate of exposure.

## 10. Final Feature Audit

In [21]:
feature_audit = pd.DataFrame({
    "feature": ["tenure_band", "spend_tier", "is_at_risk", "revenue_at_risk"],
    "dtype": [
        str(df["tenure_band"].dtype),
        str(df["spend_tier"].dtype),
        str(df["is_at_risk"].dtype),
        str(df["revenue_at_risk"].dtype)
    ],
    "null_count": [
        df["tenure_band"].isna().sum(),
        df["spend_tier"].isna().sum(),
        df["is_at_risk"].isna().sum(),
        df["revenue_at_risk"].isna().sum()
    ],
    "unique_values": [
        df["tenure_band"].nunique(),
        df["spend_tier"].nunique(),
        df["is_at_risk"].nunique(),
        df["revenue_at_risk"].nunique()
    ]
})

feature_audit

,feature,dtype,null_count,unique_values
0,tenure_band,category,0,6
1,spend_tier,category,0,5
2,is_at_risk,int64,0,2
3,revenue_at_risk,float64,0,138


### Final Dataset Validation

In [22]:
final_features = ["customerID", "tenure_band", "spend_tier", "is_at_risk", "revenue_at_risk"]

assert len(df) == 7043, f"Unexpected row count: {len(df)}"
assert df["customerID"].is_unique, "customerID uniqueness violated"
assert df[final_features].isna().sum().sum() == 0, "Nulls found in final features"

print("Final feature validation passed.")

Final feature validation passed.


## 11. Export — `telco_features.csv`

In [23]:
output_columns = ["customerID", "tenure_band", "spend_tier", "is_at_risk", "revenue_at_risk"]

df[output_columns].to_csv(
    "../data/processed/telco_features.csv",
    index=False
)

print("Export complete: ../data/processed/telco_features.csv")

Export complete: ../data/processed/telco_features.csv


### Reload & Verify

In [24]:
features = pd.read_csv("../data/processed/telco_features.csv")

print(f"Shape: {features.shape}")
assert features.shape == (7043, 5), f"Unexpected export shape: {features.shape}"
assert features.isna().sum().sum() == 0, "Nulls found in exported file"

print("Export validation passed.")
print()
print(features.isna().sum().rename("null_count"))
features.head()

Shape: (7043, 5)
Export validation passed.

customerID         0
tenure_band        0
spend_tier         0
is_at_risk         0
revenue_at_risk    0
Name: null_count, dtype: int64


,customerID,tenure_band,spend_tier,is_at_risk,revenue_at_risk
0,7590-VHVEG,0-10,10-30,0,0.0
1,5575-GNVDE,30-40,50-70,0,0.0
2,3668-QPYBK,0-10,50-70,0,0.0
3,7795-CFOCW,40-50,30-50,0,0.0
4,9237-HQITU,0-10,70-90,0,0.0


## 12. Phase 3 Conclusion

### Engineered Features

| Feature             | Type        | Purpose                                       |
|---------------------|-------------|-----------------------------------------------|
| `tenure_band`       | Categorical | Customer lifecycle segmentation               |
| `spend_tier`        | Categorical | Monthly-charge spending segmentation          |
| `is_at_risk`        | Binary      | Identify current retention opportunities      |
| `revenue_at_risk`   | Numeric     | Quantify current monthly revenue exposure     |

### Supporting Analysis

`historical_risk_segment` — validated that selected risk criteria were historically
associated with materially elevated churn (~2.67× overall rate).

### Key Results

- **174** active at-risk customers
- **2.47%** of the total customer base
- **13,933.80** current monthly recurring revenue exposure

### Output

`data/processed/telco_features.csv` — (7,043 rows × 5 columns)

### Phase Status

**Phase 3 — Feature Analysis: Complete**

All four engineered features were validated at the customer level and exported
for downstream SQL, Excel, and Power BI analysis.